# Занятие 7. Автоэнкодеры (AE) и латентные пространства

## Цели занятия:
- Изучить архитектуру Автоэнкодера (Encoder-Decoder)
- Научиться визуализировать латентное пространство
- Реализовать Denoising Autoencoder (DAE)
- Понять применение AE для снижения размерности

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import random
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Шаг 1. Загрузка данных (MNIST)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

---
## Шаг 2. Реализация простого AE

In [ ]:
class Autoencoder(nn.Module):
    """Simple Autoencoder for MNIST.
    
    Architecture:
    - Encoder: 784 -> 128 -> latent_dim
    - Decoder: latent_dim -> 128 -> 784
    """
    
    def __init__(self, latent_dim=16):
        super().__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid()  # Output in [0, 1]
        )
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)
    
    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

# Create model
model = Autoencoder(latent_dim=16).to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Шаг 3. Обучение AE

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

history = {'loss': []}

print("="*50)
print("Training Autoencoder")
print("="*50)

for epoch in range(10):
    model.train()
    running_loss = 0.0
    
    for images, _ in train_loader:
        images = images.to(device)
        
        optimizer.zero_grad()
        outputs, _ = model(images)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    history['loss'].append(avg_loss)
    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}")

---
## Шаг 4. Визуализация реконструкции

In [ ]:
model.eval()
with torch.no_grad():
    sample, _ = next(iter(test_loader))
    sample = sample.to(device)
    reconstructed, _ = model(sample)

# Plot original vs reconstructed
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    # Original
    axes[0, i].imshow(sample[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=12)
    
    # Reconstructed
    axes[1, i].imshow(reconstructed[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstructed', fontsize=12)

plt.suptitle('Autoencoder Reconstruction', fontsize=14)
plt.tight_layout()
plt.savefig('ae_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Шаг 5. Визуализация Латентного Пространства (PCA)

In [ ]:
from sklearn.decomposition import PCA

# Get latent vectors for test set
model.eval()
all_z = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        _, z = model(images)
        all_z.append(z.cpu())
        all_labels.append(labels)

all_z = torch.cat(all_z).numpy()
all_labels = torch.cat(all_labels).numpy()

print(f"Latent vectors shape: {all_z.shape}")

In [ ]:
# PCA to 2D
pca = PCA(n_components=2)
z_2d = pca.fit_transform(all_z)

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(z_2d[:, 0], z_2d[:, 1], c=all_labels, cmap='tab10', s=2, alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Latent Space Visualization (PCA)')
plt.savefig('ae_latent_pca.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Шаг 6. Denoising Autoencoder (DAE)

In [ ]:
# Add noise to images
def add_noise(images, noise_factor=0.5):
    """Add Gaussian noise to images."""
    noisy = images + noise_factor * torch.randn_like(images)
    return torch.clamp(noisy, 0, 1)

# Visualize noise effect
sample, _ = next(iter(test_loader))
noisy_sample = add_noise(sample, noise_factor=0.5)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    axes[0, i].imshow(sample[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Original' if i == 0 else '')
    
    axes[1, i].imshow(noisy_sample[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Noisy' if i == 0 else '')

plt.suptitle('Adding Noise to Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Train DAE
model_dae = Autoencoder(latent_dim=16).to(device)
optimizer_dae = torch.optim.Adam(model_dae.parameters(), lr=0.001)

print("="*50)
print("Training Denoising Autoencoder")
print("="*50)

for epoch in range(10):
    model_dae.train()
    running_loss = 0.0
    
    for images, _ in train_loader:
        images = images.to(device)
        noisy_images = add_noise(images, noise_factor=0.5).to(device)
        
        optimizer_dae.zero_grad()
        outputs, _ = model_dae(noisy_images)
        loss = criterion(outputs, images)  # Target is clean image!
        loss.backward()
        optimizer_dae.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}")

In [ ]:
# Test DAE
model_dae.eval()
with torch.no_grad():
    sample, _ = next(iter(test_loader))
    sample = sample.to(device)
    noisy_sample = add_noise(sample, noise_factor=0.5)
    denoised, _ = model_dae(noisy_sample)

# Plot
fig, axes = plt.subplots(3, 10, figsize=(15, 5))

for i in range(10):
    axes[0, i].imshow(sample[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(noisy_sample[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    axes[2, i].imshow(denoised[i].cpu().squeeze(), cmap='gray')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Noisy', fontsize=12)
axes[2, 0].set_ylabel('Denoised', fontsize=12)

plt.suptitle('Denoising Autoencoder Results', fontsize=14)
plt.tight_layout()
plt.savefig('dae_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Домашнее задание

1. Реализовать сверточный AE (Conv2d + ConvTranspose2d)
2. Сравнить качество реконструкции при latent_dim = 2, 16, 64
3. Поэкспериментировать с уровнем шума для DAE

---
## Контрольные вопросы

1. Зачем нужна функция активации Sigmoid на выходе декодера?
2. Что произойдет, если размерность латентного пространства равна входной?
3. Как AE связан с методом главных компонент (PCA)?
4. Почему DAE устойчивее к шуму чем обычный AE?